In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

REPO = Path.cwd()
if not (REPO / "downstream_tasks").exists():
    REPO = Path("/home/jovyan/dpanc/GENA_LM/GENA_LM_expression_branch")

BENCHMARK_ROOT = Path("/home/jovyan/dpanc/benchmarking")
DATA = BENCHMARK_ROOT / "data"
GENA_ROOT = BENCHMARK_ROOT / "GENA_LM"
ALPHAGENOME_ROOT = BENCHMARK_ROOT / "AlphaGenome"

sys.path.insert(0, str(REPO / "downstream_tasks/expression_prediction/gena_lm_benchmark/scripts"))
from score_ct_specificity import score_predictions

import json


In [2]:
split = "test"       # valid or test
gena_model = "dev_loss"
ag_len_window = "1Mb"

GENA_PRED_PATH = GENA_ROOT / "predictions_results" / gena_model / f"gena_lm_{split}_json812_predictions.csv"
AG_PRED_PATH = ALPHAGENOME_ROOT / "predictions" / f"results-seq_len{ag_len_window}_json812" / f"alphagenome_supported_predictions_{split}_intervals.csv"
TRUE_PATH = DATA / "borzoi_all_ids_qnorm_matrix.csv"
ONTOLOGY_JSON = ALPHAGENOME_ROOT / "data/alphagenome_track_to_qnorm_ids_all.json"

print("GENA:", GENA_PRED_PATH)
print("AG:", AG_PRED_PATH)
print("GT:", TRUE_PATH)
print("ontology:", ONTOLOGY_JSON)


GENA: /home/jovyan/dpanc/benchmarking/GENA_LM/predictions_results/dev_loss/gena_lm_test_json812_predictions.csv
AG: /home/jovyan/dpanc/benchmarking/AlphaGenome/predictions/results-seq_len1Mb_json812/alphagenome_supported_predictions_test_intervals.csv
GT: /home/jovyan/dpanc/benchmarking/data/borzoi_all_ids_qnorm_matrix.csv
ontology: /home/jovyan/dpanc/benchmarking/AlphaGenome/data/alphagenome_track_to_qnorm_ids_all.json


In [3]:
gena_pred = pd.read_csv(GENA_PRED_PATH)
ag_pred = pd.read_csv(AG_PRED_PATH)
true_raw = pd.read_csv(TRUE_PATH)

with open(ONTOLOGY_JSON) as f:
    ontology_to_ids = json.load(f)

for df in [gena_pred, ag_pred]:
    if "gene_id" not in df.columns:
        df.rename(columns={df.columns[0]: "gene_id"}, inplace=True)

print("gena_pred:", gena_pred.shape)
print("ag_pred:", ag_pred.shape)
print("true_raw:", true_raw.shape)
print("ontologies:", len(ontology_to_ids))
display(gena_pred.head())
display(ag_pred.head())
display(true_raw.head())


gena_pred: (2781, 813)
ag_pred: (2743, 286)
true_raw: (433, 24763)
ontologies: 285


,gene_id,ENCFF003KHL,ENCFF003QOJ,ENCFF007QAS,ENCFF007UXU,ENCFF009MEF,ENCFF010UKB,ENCFF010XLY,ENCFF011DHD,ENCFF012XRF,...,ENCFF993TGD,ENCFF994ERY,ENCFF994TIH,ENCFF995LYR,ENCFF996NCW,ENCFF996QEJ,ENCFF997ZTN,ENCFF998VKC,ENCFF999JNH,ENCFF999KLY
0,ENSG00000232721.2,-3.781250,-0.902344,-2.453125,-2.078125,-2.750000,-2.250000,-2.515625,-3.500000,-2.937500,...,-2.203125,-3.078125,-2.296875,-3.921875,-2.906250,-3.187500,-2.390625,-2.406250,-2.156250,-2.546875
1,ENSG00000263590.2,-4.343750,-0.726562,-3.171875,-2.203125,-2.203125,-1.921875,-2.671875,-3.687500,-3.203125,...,-3.234375,-3.500000,-2.218750,-4.406250,-3.484375,-3.828125,-2.296875,-2.343750,-1.906250,-2.875000
2,ENSG00000234277.2,-3.390625,-0.294922,-2.515625,-2.765625,-3.125000,-2.765625,-2.921875,-3.015625,-2.921875,...,-2.468750,-3.312500,-2.765625,-3.468750,-3.093750,-3.187500,-2.625000,-2.609375,-2.625000,-3.062500
3,ENSG00000181450.17,-0.404297,-0.597656,-0.017944,0.155273,-0.250000,0.014709,0.240234,-0.439453,-0.451172,...,-0.103516,-0.314453,0.089844,-0.554688,-0.277344,-0.558594,-0.145508,-0.019409,-0.001274,0.156250
4,ENSG00000143740.14,0.542969,0.968750,0.703125,0.777344,0.949219,0.750000,0.703125,0.808594,0.601562,...,0.652344,0.621094,0.494141,0.507812,0.476562,0.707031,0.660156,0.617188,0.593750,0.632812


,gene_id,ENCFF572SPV,ENCFF998VKC,ENCFF033KAE,ENCFF977WIW,ENCFF413SRE,ENCFF077ICY,ENCFF046TTE,ENCFF591MXM,ENCFF653DJK,...,ENCFF011DHD,ENCFF051CMQ,ENCFF118PSU,ENCFF589IWR,ENCFF789BAL,ENCFF523DOF,ENCFF113MLN,ENCFF014BZI,ENCFF012XRF,ENCFF496RMH
0,ENSG00000232721.2,0.062544,0.000670,0.001911,0.005827,NaN,0.000647,0.011026,0.001415,0.001745,...,0.007902,0.001461,0.000584,0.001114,NaN,NaN,NaN,NaN,0.001581,0.016813
1,ENSG00000263590.2,0.003572,0.000185,0.000506,0.001693,NaN,0.000103,0.000173,0.000134,0.000163,...,0.001708,0.000736,0.000223,0.000201,NaN,NaN,NaN,NaN,0.001315,0.000558
2,ENSG00000234277.2,0.000185,0.000045,0.000013,0.000242,NaN,0.000063,0.000345,0.000036,0.000053,...,0.000140,0.000118,0.000042,0.000044,NaN,NaN,NaN,NaN,0.000044,0.000953
3,ENSG00000181450.17,0.306130,0.062912,0.094598,0.157760,NaN,0.057589,0.068864,0.070154,0.069356,...,0.067564,0.064749,0.044934,0.048584,NaN,NaN,NaN,NaN,0.065572,0.029899
4,ENSG00000143740.14,0.198139,0.106485,0.205228,0.178856,NaN,0.201743,0.157970,0.143501,0.139113,...,0.145279,0.092469,0.111066,0.079822,NaN,NaN,NaN,NaN,0.181571,0.134298


,id,original_id,targets_identifier_base,strand_specificity,ENSG00000000003.14,ENSG00000000005.5,ENSG00000000419.12,ENSG00000000457.13,ENSG00000000460.16,ENSG00000000938.12,...,ENSG00000285958.1,ENSG00000285966.1,ENSG00000285971.1,ENSG00000285972.1,ENSG00000285976.1,ENSG00000285978.1,ENSG00000285982.1,ENSG00000285985.1,ENSG00000285988.1,ENSG00000285991.1
0,ENCFF035CWS,ENCSR094GVZ,ENCFF387UUZ,reverse,2.916907,0.134939,37.014871,5.174252,2.241552,2.510740,...,0.014965,0.014965,0.202225,0.014965,21.471765,0.014965,0.014965,0.014965,0.014965,0.134939
1,ENCFF329ENM,ENCFF672VYQ,ENCFF672VYQ,reverse,46.226096,0.218148,14.635592,7.994480,1.502216,1.641390,...,0.007986,0.007986,0.007986,0.218148,39.789006,0.007986,0.007986,0.007986,0.007986,0.103898
2,ENCFF761SPP,ENCSR561FEE,ENCFF917RKL,reverse,50.138959,0.025022,72.262047,2.668746,9.211493,0.025022,...,0.025022,0.283328,0.025022,0.339444,51.955047,0.025022,0.025022,0.025022,0.025022,0.025022
3,ENCFF731CJY,ENCFF168OLY,ENCFF168OLY,reverse,35.140427,0.015881,19.369247,3.827355,3.611565,0.152430,...,0.015881,0.015881,0.015881,0.015881,10.402241,0.015881,0.015881,0.015881,0.015881,0.152430
4,ENCFF242NRP,ENCFF153YEN,ENCFF153YEN,reverse,10.349852,0.044010,46.781483,3.609030,2.013845,1.359965,...,0.044010,0.044010,0.044010,0.492444,44.302556,0.044010,0.044010,0.044010,0.044010,1.809056


In [4]:
metadata_cols = ["gene_id", "id", "original_id", "targets_identifier_base", "strand_specificity"]
metadata_cols = [c for c in metadata_cols if c in true_raw.columns]
gene_cols = [c for c in true_raw.columns if c not in metadata_cols]
cell_id_col = "gene_id" if "gene_id" in true_raw.columns else "id"

true_gene_by_cell = true_raw.set_index(cell_id_col)[gene_cols].T.reset_index().rename(columns={"index": "gene_id"})

true_log = true_gene_by_cell.copy()
expr_cols = [c for c in true_log.columns if c != "gene_id"]
true_log[expr_cols] = np.log2(true_log[expr_cols].astype(float) + 1)

print("true_gene_by_cell:", true_gene_by_cell.shape)
print("true_log:", true_log.shape)
display(true_log.head())


true_gene_by_cell: (24759, 434)
true_log: (24759, 434)


id,gene_id,ENCFF035CWS,ENCFF329ENM,ENCFF761SPP,ENCFF731CJY,ENCFF242NRP,ENCFF465EDA,ENCFF152CYS,ENCFF900QOK,ENCFF708BZY,...,ENCFF164HRL,ENCFF749LPZ,ENCFF130XAM,ENCFF032LLH,ENCFF410AMW,ENCFF874MDG,ENCFF124OIZ,ENCFF690TCE,ENCFF837QWO,ENCFF332DXT
0,ENSG00000000003.14,1.969715,5.561512,5.676351,5.175542,3.504602,5.138176,5.044316,4.808661,3.619405,...,4.172287,2.285042,0.109712,0.598492,5.776276,2.701355,3.562363,2.524835,2.433919,2.498452
1,ENSG00000000005.5,0.182614,0.284690,0.035655,0.022732,0.062136,0.048007,0.039590,0.049442,0.016766,...,0.328076,0.188518,0.109712,0.021766,0.253767,0.003788,0.136205,0.074573,0.035847,1.641113
2,ENSG00000000419.12,5.248492,3.966762,6.194994,4.348321,5.578380,4.978141,4.830039,4.914244,4.722002,...,4.557047,4.903170,3.626724,4.905996,4.232873,3.343231,4.889001,4.364982,4.737505,3.386837
3,ENSG00000000457.13,2.626264,3.169040,1.875287,2.271233,2.204463,2.614061,2.682079,2.328680,2.257361,...,3.515856,1.650232,5.310228,2.721150,2.298108,1.697728,1.966542,2.203957,2.619850,2.624129
4,ENSG00000000460.16,1.696685,1.323206,3.352122,2.205257,1.591605,1.909723,1.921235,2.931143,1.462678,...,2.042436,1.043081,6.371195,3.544146,1.138023,0.888364,1.249176,1.726887,2.979770,1.512754


In [5]:
def mean_by_ontology(df, ontology_to_ids, name):
    df = df.set_index("gene_id")
    out = {}
    rows = []

    for ontology_name, cell_ids in ontology_to_ids.items():
        present_ids = [cell_id for cell_id in cell_ids if cell_id in df.columns]
        rows.append({"source": name, "ontology": ontology_name, "n_ids_in_ontology": len(cell_ids), "n_ids_present": len(present_ids)})
        if len(present_ids) == 0:
            continue
        out[ontology_name] = df[present_ids].astype(float).mean(axis=1)

    out = pd.DataFrame(out)
    out.insert(0, "gene_id", out.index)
    return out.reset_index(drop=True), pd.DataFrame(rows)


def corr_summary(true_df, pred_df):
    common_genes = true_df.index.intersection(pred_df.index)
    common_cells = true_df.columns.intersection(pred_df.columns)
    true = true_df.loc[common_genes, common_cells]
    pred = pred_df.loc[common_genes, common_cells]

    cell_corrs = []
    for cell in common_cells:
        true_vec = true[cell].astype(float).values
        pred_vec = pred[cell].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) >= 2 and np.std(true_vec) > 0 and np.std(pred_vec) > 0:
            cell_corrs.append(np.corrcoef(true_vec, pred_vec)[0, 1])

    gene_corrs = []
    for gene in common_genes:
        true_vec = true.loc[gene].astype(float).values
        pred_vec = pred.loc[gene].astype(float).values
        mask = np.isfinite(true_vec) & np.isfinite(pred_vec)
        true_vec = true_vec[mask]
        pred_vec = pred_vec[mask]
        if len(true_vec) >= 4 and np.std(true_vec) > 0 and np.std(pred_vec) > 0:
            gene_corrs.append(np.corrcoef(true_vec, pred_vec)[0, 1])

    return {"corr_genes": float(np.mean(cell_corrs)) if cell_corrs else np.nan, "corr_cells": float(np.mean(gene_corrs)) if gene_corrs else np.nan, "n_cells": len(common_cells), "n_genes": len(common_genes)}


In [6]:
gena_ontology, gena_summary = mean_by_ontology(gena_pred, ontology_to_ids, "GENA")
ag_ontology, ag_summary = mean_by_ontology(ag_pred, ontology_to_ids, "AG")
true_ontology, true_summary = mean_by_ontology(true_log, ontology_to_ids, "GT_log2p")

ontology_summary = pd.concat([gena_summary, ag_summary, true_summary], ignore_index=True)

print("gena_ontology:", gena_ontology.shape)
print("ag_ontology:", ag_ontology.shape)
print("true_ontology:", true_ontology.shape)
display(ontology_summary.head())
display(gena_ontology.head())


gena_ontology: (2781, 286)
ag_ontology: (2743, 286)
true_ontology: (24759, 208)


,source,ontology,n_ids_in_ontology,n_ids_present
0,GENA,CL:0000047 polyA plus RNA-seq,2,2
1,GENA,CL:0000062 total RNA-seq,1,1
2,GENA,CL:0000084 total RNA-seq,3,3
3,GENA,CL:0000115 total RNA-seq,1,1
4,GENA,CL:0000121 polyA plus RNA-seq,1,1


,gene_id,CL:0000047 polyA plus RNA-seq,CL:0000062 total RNA-seq,CL:0000084 total RNA-seq,CL:0000115 total RNA-seq,CL:0000121 polyA plus RNA-seq,CL:0000127 total RNA-seq,CL:0000134 polyA plus RNA-seq,CL:0000137 total RNA-seq,CL:0000138 total RNA-seq,...,UBERON:0009834 total RNA-seq,UBERON:0010414 total RNA-seq,UBERON:0011907 total RNA-seq,UBERON:0015143 total RNA-seq,UBERON:0018115 polyA plus RNA-seq,UBERON:0018116 polyA plus RNA-seq,UBERON:0018117 polyA plus RNA-seq,UBERON:0018118 polyA plus RNA-seq,UBERON:0036149 total RNA-seq,UBERON:1000010 polyA plus RNA-seq
0,ENSG00000232721.2,-2.695312,-2.406250,-2.968750,-2.406250,-2.296875,-2.460938,-2.429688,-2.546875,-2.468750,...,-3.404297,-2.957031,-2.863281,-3.140625,-3.911458,-3.859375,-3.848958,-3.765625,-2.949219,-0.812500
1,ENSG00000263590.2,-3.281250,-2.343750,-3.213542,-2.796875,-2.781250,-2.429688,-2.546875,-2.578125,-2.546875,...,-3.570312,-3.550781,-3.488281,-3.859375,-4.427083,-4.375000,-4.364583,-4.291667,-3.152344,-0.695312
2,ENSG00000234277.2,-2.835938,-2.609375,-2.958333,-3.109375,-2.234375,-2.843750,-2.851562,-2.875000,-2.984375,...,-2.944661,-2.906250,-2.679688,-3.046875,-3.421875,-3.398438,-3.416667,-3.369792,-2.921875,-0.341797
3,ENSG00000181450.17,0.129639,-0.019409,0.108561,-0.043457,0.086426,-0.034546,-0.157715,-0.036133,-0.160156,...,-0.423503,-0.614258,-0.517578,-0.679688,-0.389323,-0.430176,-0.401693,-0.402344,-0.403809,-0.511719
4,ENSG00000143740.14,0.718750,0.617188,0.921875,0.734375,0.406250,0.644531,0.599609,0.652344,0.621094,...,0.802734,0.470215,0.659180,0.476562,0.501302,0.476562,0.533854,0.472005,0.627930,0.832031


In [8]:
gena_i = gena_ontology.set_index("gene_id")
ag_i = ag_ontology.set_index("gene_id")
true_i = true_ontology.set_index("gene_id")

common_genes = true_i.index.intersection(gena_i.index).intersection(ag_i.index)
common_ontologies = true_i.columns.intersection(gena_i.columns).intersection(ag_i.columns)

true_aligned = true_i.loc[common_genes, common_ontologies]
gena_aligned = gena_i.loc[common_genes, common_ontologies]
ag_aligned = ag_i.loc[common_genes, common_ontologies]

corr_result = pd.DataFrame([
    {"model": "GENA", **corr_summary(true_aligned, gena_aligned)},
    {"model": "AlphaGenome", **corr_summary(true_aligned, ag_aligned)},
])

display(corr_result)


,model,corr_genes,corr_cells,n_cells,n_genes
0,GENA,0.713612,0.400919,207,2743
1,AlphaGenome,0.705851,0.303044,207,2743
